# Finding and Analzying Securities for Supporting Investment Decisions for 2025

## Introduction
The aim of this project is to gather information from the internet about suggested stocks, and analyse them.
**Objectives**:
- Define short-term investmen
- Use a webscraper to collect information about ssuggested securities
- Store the collected data in a database
- Select the 3 most frequently mentioned securities
- Prepare data for analysis
- Analyse data
- Summarize insights

### What are the benefits of investing in growth stocks?
Growth stocks are shares of companies that are expected to grow significantly faster than the overall market or their industry peers. These companies typically reinvest their earnings into the business to fuel further expansion, which often leads to impressive revenue and profit growth over time.1

### What will the analysis be focus on?
The analysis focuses on:
- visualize the most frequent words used when mentioning the specific stocks
- analyze the context around the stock's company; is it mostly positive or negative? Probably we don't want to invest in a company which has a negative publicity
- *volatility*: check the difference between the highest and lowest values within a day. Also, examine the difference between the opening and closing prices on the same day, and compare the opening price with the previous day's closing price to see the mood changes while the market is closed.
- *monotonity*: analyze the closing prices to determine the overall trend of the security

# SQL Database
Ceate a database to store data which will be analysed later. 

Considerations:
- Rarely, but it can happen that one company has more tickers
- When looking for tickers, it may not be possible to scrape company name and sector from the texts at first, so those are optional, can be filled with later searches
- Context is text scraped from where the company and/or sector was mentioned if exists
- Link is the URL of the website where the ccompany and/or sector was mentioned if exists

#### Tables

**stock_info**  
Ticker - text, primary key  
Company name - optional text,  
Sector - optional text,
Investopedia_link - optional text
  
**company_context**  
Company - text, primary key  
Context - optional text,  
link - optional text  

**sector_context**  
Sector - text, primary key  
Context - optional text,  
Link - optional text  






## Create tables

In [598]:
import sqlite3 as sql

# To avoid conflict with different versions
%config SqlMagic.style = '_DEPRECATED_DEFAULT'
%reload_ext sql

In [599]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE IF NOT EXISTS stock_info(
    ticker VARCHAR(10) NOT NULL UNIQUE,
    company TEXT,
    sector VARCHAR(100),
    investopedia_link TEXT,
    PRIMARY KEY (ticker),
    FOREIGN KEY (company) REFERENCES company_context(company),
    FOREIGN KEY (sector) REFERENCES sector_context(sector)
);

[]

In [597]:
%%sql sqlite:///stockAnalysis.db
CREATE TABLE IF NOT EXISTS company_context(
    company TEXT NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (company)
);

[]

In [600]:
%%sql sqlite://sqlite:///stockAnalysis.db
CREATE TABLE  IF NOT EXISTS sector_context(
    sector VARCHAR(100) NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (sector)
);

##### Show created tables

In [601]:
%%sql sqlite:///stockAnalysis.db
SELECT name
FROM sqlite_master
WHERE type='table';

name
stock_info
company_context
sector_context


## Handle Database queries

In [637]:
from io import StringIO
from contextlib import redirect_stdout

class stock_analysis_db:
    debug = None

    def __init__(self, mock_db = False, debug = False):
        self.debug = debug

        # If db is set, use memory for mock database
        if mock_db:
            %sql sqlite:///:memory:
        else:
            %sql sqlite:///stockAnalysis.db
        
    def validate_dict_raise(self, data):
        if not (isinstance(data, list) and all(isinstance(row, dict) for row in data)):
            raise TypeError("Data is not a list of dictionaries")

    def insert_into(self, table, data):
        try:
            # data must be a list of dictionaries
            self.validate_dict_raise(data)

            queries = []
            for row in data:
                columns = ", ".join(row.keys())    
                values = ", ".join(
                    f"'{v}'" if isinstance(v, str) and v!= "N/A" else "NULL"
                    for v in row.values()
                )
                query = f"INSERT INTO {table} ({columns}) VALUES({values});" 
                
                # sql magic doesn't throw just writes to stdout
                output = StringIO()
                with redirect_stdout(output):
                    %sql {query}
                msg = output.getvalue()
                if self.debug:
                    print(msg)
                
                if 'sqlite3.OperationalError' in msg:
                    raise sql.OperationalError(msg)

                if 'sqlite3.IntegrityError' in msg:
                    raise sql.IntegrityError(msg)
        
        except TypeError as e:
            print(e)
        except sql.OperationalError as e:
            print(f"SQL insert into Failed due to operation error: {e}")
            raise

        except sql.IntegrityError as e:
            print(f"SQL insert into failed due to integrity error: {e}")
            raise

    def query_smtg(self):
        res = %sql SELECT * FROM table
        return res

# Gather data
I will gather data from the Internet using a Webscraper:
- Tickers mentioned as good investment oportunities, whenever possible I will also save the company and sector
- I will save context from atricles about the company, and the sector

#### Ethical Considerations
...

## Webscraper
- Gather data about AI stocks from a website
- Identify the tickers in the artices
- Store data gathered in a database


In [603]:
# Merge later once tested enough
import re
from urllib.request import urlopen
from urllib.request import Request
from urllib.error import HTTPError, URLError
# from urllib.parse import urljoin
from bs4 import BeautifulSoup
import time

# base_url = "https://www.mintos.com/blog/artificial-intelligence-stocks/"
# base_url = "https://money.usnews.com/investing/articles/up-and-coming-stocks-to-buy-in-2025"
base_url = "https://www.bing.com/search?q=best+stocks+to+invest+in+2025"
base_url = "https://www.investopedia.com/the-best-growth-stocks-8783031"


In [604]:
bs = get_bs_from_url()
# print(bs)

In [605]:
class webscraper:
    debug = False

    def get_bs_from_url(url = base_url):
        try:
            headers = {
                "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36",
                # "Referer": "https://www.google.com/", # simulate Google search
                #"Connection": "keep-alive",
                #"Upgrade-Insecure-Requests": "1"
            }
            
            # To avoid 'Too many requests'
            time.sleep(5)
            
            response = urlopen(Request(url, headers=headers))
            return BeautifulSoup(response.read(), 'html.parser')

        except HTTPError as e:
            print(f"HTTP error: {e.code} - {e.reason}")
            return None
        
        except URLError as e:
            print(f"URL error: {e.reason}")
            return None 
    
    def scrape_ticker_data_from_investopedia(self, soup):
        bs = get_bs_from_url("https://www.investopedia.com/the-best-growth-stocks-8783031")
        if not soup:
            return []

       # Keressük azt a táblázatot, amelynek a fejlécében szerepel a "Stock" szó
        table = None
        for tbl in soup.find_all("table"):
            if tbl.find("th", string=lambda s: s and "stock" in s.lower()):
                table = tbl
                break

        if table:
            # First row of the table body contains the indexes
            first_row = table.find("tbody").find("tr")
            column_name_indexes = {}
            for i, td in enumerate(first_row.find_all("td", recurisive=False)):
                text = td.get_text(strip=True).lower()
                if "ticker" in text:
                    column_name_indexes["ticker"] = i
                elif "company" in text:
                    column_name_indexes["company"] = i
                elif "sector" in text:
                    column_name_indexes["sector"] = i
                i+=1

            all_ticker_data = []
            # Get data from table
            for row in first_row.find_next_siblings("tr"):
                ticker_data = {}
                cells = row.find_all("td")
                
                # Ticker has a link to the profile
                link = cells[column_name_indexes["ticker"]].find('a')
                if link:
                    ticker_data["investopedia_link"] = link.attrs['href']         
                
                # Extract data from each cell
                ticker_data["ticker"] = cells[column_name_indexes["ticker"]].get_text(strip=True)
                ticker_data["company"] = cells[column_name_indexes["company"]].get_text(strip=True)
                ticker_data["sector"] = cells[column_name_indexes["sector"]].get_text(strip=True)

                all_ticker_data.append(ticker_data)

            return all_ticker_data

In [607]:
my_scraper = webscraper()
ret = my_scraper.scrape_ticker_data_from_investopedia(bs)

## Data preparation
First, I analyze the data and check the following:
- Validate if the expected columns are present
- Validate data formats and their defined domains
- Handle any missing data
- Fill NaN values using a custom strategy
- inspect zero values if any

# Unit Tests

### Create mock tables in memory database

In [608]:
# To avoid conflict with different versions
%config SqlMagic.style = '_DEPRECATED_DEFAULT'

In [609]:
%%sql sqlite:///:memory:
CREATE TABLE IF NOT EXISTS stock_info(
    ticker VARCHAR(10) NOT NULL UNIQUE,
    company TEXT,
    sector VARCHAR(100),
    investopedia_link TEXT,
    PRIMARY KEY (ticker),
    FOREIGN KEY (company) REFERENCES company_context(company),
    FOREIGN KEY (sector) REFERENCES sector_context(sector)
);

[]

In [610]:
%%sql sqlite:///:memory:
CREATE TABLE IF NOT EXISTS company_context(
    company TEXT NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (company)
);

[]

In [611]:
%%sql sqlite:///:memory:
CREATE TABLE  IF NOT EXISTS sector_context(
    sector VARCHAR(100) NOT NULL UNIQUE,
    context TEXT,  
    link TEXT, 
    PRIMARY KEY (sector)
);

[]

In [653]:
import unittest
from io import StringIO
from unittest.mock import patch
import sqlite3 as sql

class TestDatabase(unittest.TestCase):

    def setUp(self):
        # Suppress stdout automatically
        self.patcher = patch('sys.stdout', new_callable=StringIO)
        self.mock_stdout = self.patcher.start()

    def tearDown(self):
        # Restore stdout after test
        self.patcher.stop()

    # Define test type when we expect no exception
    def assertNotRaises(self, exception, func, *args, **kwargs):
        try:
            func(*args, **kwargs)
        except exception as e:
            self.fail(f"Expected no {exception}, but got: {e}")
    
    def test_db_class_exists(self):
        self.assertIsNotNone(stock_analysis_db)

    # Test insert data validator class

    def test_validate_dict_raise_exists(self):
        self.assertIsNotNone(stock_analysis_db.validate_dict_raise)

    def test_validate_dict_raise_exception_empty_1(self):
        test_data = None
        with self.assertRaises(TypeError):
            stock_analysis_db.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_2(self):
        test_data = ""
        with self.assertRaises(TypeError):
            stock_analysis_db.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_3(self):
        test_data = []
        with self.assertRaises(TypeError):
            stock_analysis_db.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_empty_4(self):
        test_data = {}
        with self.assertRaises(TypeError):
            stock_analysis_db.validate_dict_raise(test_data)

    def test_validate_dict_raise_exception_not_list(self):
        test_dict = {'col1': 'link',
                        'col2': 'XXX',
                        'col3': 'text',
                        'col4': 'text'}
        with self.assertRaises(TypeError):
             stock_analysis_db.validate_dict_raise(test_dict)

    def test_validate_dict_raise_exception_not_dict_in_list(self):
        test_list = ['text1', 'text2', 'text3', 'text4']
        with self.assertRaises(TypeError):
            stock_analysis_db.validate_dict_raise(test_list)


    def test_validate_dict_not_raise_exception(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        db_instance = stock_analysis_db(mock_db=True)
        self.assertNotRaises(TypeError, db_instance.validate_dict_raise, my_test_list_dict)

    # Test insert_into function
    
    def test_insert_into_exists(self):
        self.assertIsNotNone(stock_analysis_db.insert_into)

    def test_insert_into_wrong_table(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        db_instance = stock_analysis_db(mock_db=True)
        with self.assertRaisesRegex(sql.OperationalError, r"syntax error"):
            db_instance.insert_into("table", my_test_list_dict)

    def test_insert_into_wrong_column_name(self):
        my_test_list_dict = [{'col1': 'text', 'col2': 'XXX', 'col3': 'text', 'col4': 'text'},
                        {'col1': 'text', 'col2': 'RUA','col3': 'text', 'col4': 'text'}]
        db_instance = stock_analysis_db(mock_db=True)
        with self.assertRaisesRegex(sql.OperationalError, r"has no column named"):
            db_instance.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_no_primary_key(self):
        my_test_list_dict = [{'investopedia_link': 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=TEM',
                              'company': 'Tempus AI Inc.',
                              'sector': 'Health Care'},]
        db_instance = stock_analysis_db(mock_db=True)
        with self.assertRaisesRegex(sql.IntegrityError, r"constraint failed"):
            db_instance.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_wrong_key_type(self):
        my_test_list_dict = [{'investopedia_link': 'https://www.investopedia.com/markets/quote?tvwidgetsymbol=TEM',
                              'ticker': 3,
                              'company': 'Tempus AI Inc.',
                              'sector': 'Health Care'},]
        db_instance = stock_analysis_db(mock_db=True)
        with self.assertRaisesRegex(sql.IntegrityError, r"constraint failed"):
            db_instance.insert_into("stock_info", my_test_list_dict)

    def test_insert_into_correct(self):
        my_test_list_dict = [{'investopedia_link': 'link',
                              'ticker': 'XXX',
                              'company': 'company',
                              'sector': 'sector'},]
        db_instance = stock_analysis_db(mock_db=True)
        db_instance.insert_into("stock_info", my_test_list_dict)

        res = %sql "SELECT investopedia_link,ticker,company,sector FROM stock_info WHERE ticker = 'XXX'"

        # Verify and delete data
        self.assertEqual(res[0][0], 'link')
        self.assertEqual(res[0][1], 'XXX')
        self.assertEqual(res[0][2], 'company')
        self.assertEqual(res[0][3], 'sector')

        %sql "DELETE FROM stock_info WHERE ticker='XXX'"
        
unittest.main(argv=['ingored', '-v'], exit=False)

test_db_class_exists (__main__.TestDatabase.test_db_class_exists) ... ok
test_insert_into_correct (__main__.TestDatabase.test_insert_into_correct) ... ok
test_insert_into_exists (__main__.TestDatabase.test_insert_into_exists) ... ok
test_insert_into_no_primary_key (__main__.TestDatabase.test_insert_into_no_primary_key) ... ok
test_insert_into_wrong_column_name (__main__.TestDatabase.test_insert_into_wrong_column_name) ... ok
test_insert_into_wrong_key_type (__main__.TestDatabase.test_insert_into_wrong_key_type) ... ok
test_insert_into_wrong_table (__main__.TestDatabase.test_insert_into_wrong_table) ... ok
test_validate_dict_not_raise_exception (__main__.TestDatabase.test_validate_dict_not_raise_exception) ... ok
test_validate_dict_raise_exception_empty_1 (__main__.TestDatabase.test_validate_dict_raise_exception_empty_1) ... ok
test_validate_dict_raise_exception_empty_2 (__main__.TestDatabase.test_validate_dict_raise_exception_empty_2) ... ok
test_validate_dict_raise_exception_empty_3 (

## Analysis

## Conclusions
- META has been the least volatile and demonstrated the most stable growth throughout the year
- LLY had low volatility and steady growth, though it started to show signs of decline towards the end of the year. It would make sense to analyze an additional year of data to better understand its long-term trend
- SHOP showed high volatility with large swings in price, and an upward trend during the last months
- NOW appears as a riskier investment choice, with more volatility and fluctuations in its performance
- RIOT showed a consistent decline throughout the year, indicating a potentially negative trend

## Possible improvements for the project
1. A web-scraper might be added to the pipeline which analyses articles of well-known websites that suggest financial assets for short-term investments.
2. The pipeline can be automated to pull data from the web scraper and evaluate performances, for example, daily
3. I’m only using stocks now, but it can be expanded with other types of securities like bonds, ETFs, etc.
4. Different short-term strategies can be explored and built into the code.
5. I might explore other data sources to clean and validate the data

## Difficulties with data
The data is well-structured, and for the one-year period analyzed, there were no major issues with missing or empty values. Handling multi-indexes was a bit more challenging, but after some practice, it became easier to manage.

## References
$^1$: https://www.investopedia.com/terms/s/shorterminvestments.asp  
$^2$: https://www.forbes.com/sites/investor-hub/article/best-growth-stocks/  
$^3$: https://pypi.org/project/yfinance/

In [15]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
